In [1]:
import asyncio
from picoagents import Agent
from picoagents.llm import OpenAIChatCompletionClient
import os
from dotenv import load_dotenv

In [2]:
# Setup the language model client
load_dotenv("/home/hemamgholizadeh/multiagents/picoagent/.env")

client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY")
)


In [3]:
poet = Agent(
    name="poet",
    description="Haiku poet. ",
    instructions="You are a haiku poet. ",
    model_client=client
)

In [4]:
async def test_poet():
    response = await poet.run("Write a haiku about cherry bloosoms in spring")
    print(f"Poet says: {response}")

await test_poet()

Poet says: [user] 07:37:29 | Write a haiku about cherry bloosoms in spring
[poet] 07:37:32 | Petals dance on breeze,  
Whispers of pink fill the air,  
Spring's soft breath awakes.

[usage] duration: 2.5s, tokens: in:30, out:23 | finish: stop


In [5]:
critic = Agent(
    name="critic",
    description="Poetry critic who provides constructive feedback on haikus.",
    instructions="You are a hauku critic. \
        suggestions for environment. Be constructive and brief. \
            If satisfied with the hauku, respnd with 'APPROVED'",
    model_client=client
)

In [6]:
async def test_critic():
    haiku = ("Petals drift like snow,  \
        Whispers of spring fill the air—  \
        Pink clouds grace the sky.")
    response = await critic.run(f"Please critique this haiku: {haiku}")
    print(f"Critic says: {response}")
await test_critic()

Critic says: [user] 07:37:32 | Please critique this haiku: Petals drift like snow,          Whispers of spring fill the air—          Pink clouds grace the sky.
[critic] 07:37:34 | This haiku beautifully captures the essence of spring through vivid imagery. The simile of petals drifting like snow creates a lovely visual contrast. However, consider tightening the phrasing. For instance, "Pink clouds" could be more evocative if you specify what they visually imply or evoke. This would deepen the emotional resonance. Overall, a lovely piece with room for more specificity!

APPROVED

[usage] duration: 2.0s, tokens: in:72, out:78 | finish: stop


In [7]:
from picoagents.orchestration import RoundRobinOrchestrator
from picoagents.termination import MaxMessageTermination, TextMentionTermination

# create termination conditions
termination = (MaxMessageTermination(max_messages=4) |
               TextMentionTermination(text="APPROVED"))

orchestrator = RoundRobinOrchestrator(agents=[poet, critic],
                                      termination=termination,
                                      max_iterations=4)

In [8]:
# Run orchestration

async def run_orchestration():
    task="Write a haiku about cherry blossoms in spring"
    stream = orchestrator.run_stream(task)
    async for message in stream:
        print(f"{message}")
await run_orchestration()

[user] 07:37:34 | Write a haiku about cherry blossoms in spring
[poet] 07:37:36 | Petals float like dreams,  
Whispers of a soft spring breeze,  
Beauty fades too soon.
[critic] 07:37:38 | The imagery is lovely and captures the transient beauty of cherry blossoms. However, it could benefit from a more vivid connection to the spring season. Perhaps adding specific sensations or colors can enhance the experience. Consider revising to evoke the warmth or the excitement of spring in a more tangible way. 

Keep up the good work!
[poet] 07:37:39 | Cherry blooms in pink,  
Sun-warmed breezes gently sway,  
Spring's embrace unfolds.
🏁 RoundRobinOrchestrator: Cherry blooms in pink,  
Sun-warmed breezes gently sway,  
Spring's embrace unfo... | duration: 5.2s, tokens: in:358, out:105, calls: 3 . Stop reason: Composite (any): Maximum messages reached (4/4)


In [9]:
from threading import Thread
from picoagents.webui import serve

def start_webui():
    serve(entities=[orchestrator], port=8070, auto_open=False)


print("PicoAgents WebUI: http://127.0.0.1:8070")

PicoAgents WebUI: http://127.0.0.1:8070
